In [1]:
#!pip -q install kagglehub albumentations opencv-python-headless pandas tqdm

Nhớ đổi DRIVE_PROJECT_DIR

In [2]:
# =========================
# CONFIG - CHANGED
# =========================
CONFIG = {
    # Kaggle dataset
    "KAGGLE_DATASET": "arischii05/cleaned-foodseg103",
    "DATASET_FOLDER_NAME": "foodseg103_rebalanced",

    # Checkpoint/log in Drive
    "PROJECT_DIR": "/content/drive/MyDrive/tmp",
    "CKPT_DIRNAME": "checkpoints_overfit32_rebalanced",
    "LOG_CSV_NAME": "train_log_overfit32_rebalanced.csv",

    # Split
    "DRIVE_PROJECT_DIR": "/content/drive/MyDrive/tmp",
    "SPLIT_JSON_NAME": "splits_seed42.json",

    # Seed / split
    "SEED": 42,
    "VAL_RATIO": 0.10,
    "OVERFIT_MODE": True,
    "OVERFIT_SAMPLES": 32,

    # Dataset
    "NUM_CLASSES": 76,   # background 0 + foreground 1..75
    "BACKGROUND_ID": 0,

    # Colab T4 safe defaults
    "IMAGE_SIZE": 384,
    "BATCH_SIZE": 4,
    "NUM_WORKERS": 2,
    "PIN_MEMORY": True,

    # Model / transfer learning
    "MODEL_NAME": "BiSeNetV1_ResNet18_Pretrained",
    "PRETRAINED_BACKBONE": True,
    "USE_AUX_HEAD": True,
    "AUX_WEIGHT": 0.3,

    # Training
    "EPOCHS": 60,
    "LR": 0.005,
    "MOMENTUM": 0.9,
    "WEIGHT_DECAY": 5e-4,
    "POLY_POWER": 0.9,
    "USE_AMP": True,

    # Checkpoint
    "RESUME": False,
    "SAVE_BEST": True,
    "SAVE_LATEST_EVERY_EPOCH": True,

    # Memory optimization
    "GRAD_CLIP_NORM": 1.0,
}

In [3]:

# =========================
# IMPORTS
# =========================
import os
import gc
import json
import time
import math
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.auto import tqdm

import kagglehub
from google.colab import drive

# =========================
# SEED
# =========================
random.seed(CONFIG["SEED"])
np.random.seed(CONFIG["SEED"])
torch.manual_seed(CONFIG["SEED"])
torch.cuda.manual_seed_all(CONFIG["SEED"])
torch.backends.cudnn.benchmark = True

# =========================
# MOUNT DRIVE
# =========================
drive.mount("/content/drive", force_remount=True)

# =========================
# PATHS
# =========================
PROJECT_DIR = Path(CONFIG["DRIVE_PROJECT_DIR"])
CKPT_DIR = PROJECT_DIR / CONFIG["CKPT_DIRNAME"]
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# DOWNLOAD DATASET FROM KAGGLE
# =========================
kaggle_root = Path(kagglehub.dataset_download(CONFIG["KAGGLE_DATASET"]))
print("Kaggle root:", kaggle_root)

# thử tìm DATA_ROOT
candidate_1 = kaggle_root / CONFIG["DATASET_FOLDER_NAME"]
candidate_2 = kaggle_root

if candidate_1.exists():
    DATA_ROOT = candidate_1
elif candidate_2.exists():
    DATA_ROOT = candidate_2
else:
    raise FileNotFoundError("Không tìm thấy DATA_ROOT")

print("DATA_ROOT:", DATA_ROOT)

# in sơ bộ cây thư mục
print("\nTop-level files/folders:")
for p in sorted(DATA_ROOT.iterdir()):
    print(" -", p.name)

Mounted at /content/drive
Using Colab cache for faster access to the 'cleaned-foodseg103' dataset.
Kaggle root: /kaggle/input/cleaned-foodseg103
DATA_ROOT: /kaggle/input/cleaned-foodseg103/foodseg103_rebalanced

Top-level files/folders:
 - class_mapping.csv
 - class_mapping.json
 - class_rebalance_plan.csv
 - meta.json
 - test
 - train


# Kiểm tra dataset rebalanced và load mapping

In [4]:
# =========================
# VERIFY DATASET
# =========================
required_paths = [
    DATA_ROOT / "meta.json",
    DATA_ROOT / "class_mapping.csv",
    DATA_ROOT / "train" / "img",
    DATA_ROOT / "train" / "ann",
    DATA_ROOT / "test" / "img",
    DATA_ROOT / "test" / "ann",
]

for p in required_paths:
    assert p.exists(), f"Thiếu path: {p}"

import base64
import csv
import io
import zlib


def load_json_file(path: Path):
    """Load a JSON file.

    Args:
        path: Path to the JSON file.
    Returns:
        Parsed JSON content.
    Raises:
        OSError: If the file cannot be opened.
        json.JSONDecodeError: If the file is not valid JSON.
    """

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)



def load_class_mapping_rows(path: Path):
    """Load rows from class_mapping.csv.

    Args:
        path: Path to the CSV mapping file.
    Returns:
        List of CSV row dictionaries.
    Raises:
        OSError: If the file cannot be opened.
    """

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))



def find_annotation_path(ann_dir: Path, stem: str):
    """Resolve the annotation JSON path for one image stem.

    Args:
        ann_dir: Annotation directory.
        stem: Image stem without extension.
    Returns:
        Path to the annotation JSON.
    Raises:
        FileNotFoundError: If the annotation file does not exist.
    """

    ann_path = ann_dir / f"{stem}.jpg.json"
    if not ann_path.exists():
        raise FileNotFoundError(f"Cannot find annotation for stem={stem}")
    return ann_path



def decode_bitmap_array(bitmap_b64: str):
    """Decode one compressed bitmap payload into a numpy array.

    Args:
        bitmap_b64: Base64 + zlib compressed bitmap payload.
    Returns:
        Decoded bitmap as a numpy array.
    Raises:
        ValueError: If the bitmap cannot be decoded.
        OSError: If the decoded payload is not a valid image.
    """

    raw = zlib.decompress(base64.b64decode(bitmap_b64))
    bitmap = Image.open(io.BytesIO(raw))
    bitmap.load()
    return np.array(bitmap)



def rasterize_annotation_to_mask(ann_path: Path, image_hw, background_id: int = 0, num_classes: int = 76):
    """Rasterize one annotation JSON file into a dense mask.

    Args:
        ann_path: Path to the annotation JSON file.
        image_hw: Tuple of image height and width.
        background_id: Background class id.
        num_classes: Total number of classes including background.
    Returns:
        Tuple of `(mask, flags)` where `mask` is a uint8 numpy array.
    Raises:
        FileNotFoundError: If the annotation file is missing.
        json.JSONDecodeError: If the annotation JSON is malformed.
    """

    ann = load_json_file(ann_path)
    image_h, image_w = image_hw
    ann_size = ann.get("size", {})
    ann_h = int(ann_size.get("height", image_h))
    ann_w = int(ann_size.get("width", image_w))
    flags = []

    if (ann_h, ann_w) != (image_h, image_w):
        if (ann_w, ann_h) == (image_h, image_w):
            ann_h, ann_w = image_h, image_w
            flags.append("size_swapped_fixed")
        else:
            ann_h, ann_w = image_h, image_w
            flags.append("size_overridden_to_image")

    mask = np.full((ann_h, ann_w), background_id, dtype=np.uint8)

    for obj in ann.get("objects", []):
        class_id = int(obj.get("classId", background_id))
        class_title = obj.get("classTitle", "unknown")

        if class_id <= background_id or class_id >= num_classes:
            class_id = int(TARGET_TITLE_TO_ID.get(class_title, background_id))
        if class_id <= background_id or class_id >= num_classes:
            flags.append(f"skip_unknown_class:{class_title}")
            continue

        bitmap = obj.get("bitmap", {})
        if "data" not in bitmap or "origin" not in bitmap:
            flags.append(f"skip_invalid_bitmap:{class_title}")
            continue

        bitmap_arr = decode_bitmap_array(bitmap["data"])
        if bitmap_arr.ndim == 3:
            bitmap_arr = bitmap_arr[..., 0]
        bitmap_mask = bitmap_arr > 0

        ox, oy = map(int, bitmap.get("origin", [0, 0]))
        mh, mw = bitmap_mask.shape
        x1, y1 = max(0, ox), max(0, oy)
        x2, y2 = min(ann_w, ox + mw), min(ann_h, oy + mh)

        if x1 >= x2 or y1 >= y2:
            flags.append(f"skip_oob:{class_title}")
            continue

        if (x1, y1, x2, y2) != (ox, oy, ox + mw, oy + mh):
            flags.append(f"clip_oob:{class_title}")

        sx1, sy1 = x1 - ox, y1 - oy
        sx2, sy2 = sx1 + (x2 - x1), sy1 + (y2 - y1)
        cropped_bitmap = bitmap_mask[sy1:sy2, sx1:sx2]
        mask_slice = mask[y1:y2, x1:x2]
        mask_slice[cropped_bitmap] = class_id

    if mask.shape != (image_h, image_w):
        mask = np.array(Image.fromarray(mask).resize((image_w, image_h), resample=Image.NEAREST), dtype=np.uint8)
        flags.append("mask_resized_to_image")

    return mask, flags


meta = load_json_file(DATA_ROOT / "meta.json")
class_mapping_rows = load_class_mapping_rows(DATA_ROOT / "class_mapping.csv")
meta_classes = meta["classes"]

TARGET_TITLE_TO_ID = {item["title"]: int(item["id"]) for item in meta_classes}
ID_TO_CLASS = {0: "background", **{int(item["id"]): item["title"] for item in meta_classes}}
CLASS_TO_ID = dict(TARGET_TITLE_TO_ID)
BACKGROUND_ID = 0
NUM_FOREGROUND_CLASSES = len(meta_classes)
NUM_CLASSES = max(TARGET_TITLE_TO_ID.values()) + 1

assert NUM_FOREGROUND_CLASSES == 75, f"Expected 75 foreground classes, got {NUM_FOREGROUND_CLASSES}"
assert NUM_CLASSES == 76, f"Expected 76 total classes including background, got {NUM_CLASSES}"
assert sorted(TARGET_TITLE_TO_ID.values()) == list(range(1, NUM_CLASSES)), "Class ids must be contiguous from 1..75"

csv_target_ids = sorted({int(row["new_class_id"]) for row in class_mapping_rows})
assert csv_target_ids == list(range(1, NUM_CLASSES)), "class_mapping.csv is inconsistent with meta.json"

CONFIG["NUM_CLASSES"] = NUM_CLASSES
CONFIG["BACKGROUND_ID"] = BACKGROUND_ID

print("meta.json classes:", len(meta_classes))
print("class_mapping.csv rows:", len(class_mapping_rows))
print("BACKGROUND_ID:", BACKGROUND_ID)
print("NUM_FOREGROUND_CLASSES:", NUM_FOREGROUND_CLASSES)
print("NUM_CLASSES:", NUM_CLASSES)
print("Example classes:", [ID_TO_CLASS[i] for i in range(1, min(10, NUM_CLASSES))])

class_mapping.json keys: dict_keys(['schema', 'class_to_id', 'target_title_to_id', 'id_to_class', 'background_id', 'num_foreground_classes', 'num_classes', 'dropped_source_classes'])
BACKGROUND_ID: 0
NUM_CLASSES: 77
Example classes: ['apple', 'apricot', 'asparagus', 'avocado', 'banana', 'blueberry', 'bread', 'broccoli', 'cabbage']
Dropped classes: ['biscuit', 'cake', 'candy', 'chocolate', 'egg tart', 'ice cream', 'kelp', 'popcorn', 'pudding']


# Discover file stems + split train/val cố định

In [5]:
IMG_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]


def find_image_path(img_dir: Path, stem: str):
    """Resolve an image path from a stem.

    Args:
        img_dir: Directory containing image files.
        stem: File stem without extension.
    Returns:
        Matching image path or `None`.
    Raises:
        OSError: If directory iteration fails.
    """

    for ext in IMG_EXTS:
        p = img_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None



def list_image_stems(img_dir: Path):
    """List all image stems in a directory.

    Args:
        img_dir: Directory containing image files.
    Returns:
        Sorted list of unique image stems.
    Raises:
        OSError: If directory iteration fails.
    """

    stems = []
    for p in img_dir.iterdir():
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            stems.append(p.stem)
    return sorted(stems)



def annotation_path_to_stem(path: Path):
    """Convert an annotation file path to an image stem.

    Args:
        path: Annotation JSON path.
    Returns:
        Image stem without extension.
    Raises:
        ValueError: If the annotation filename format is unexpected.
    """

    name = path.name
    if name.endswith(".jpg.json"):
        return name[:-9]
    raise ValueError(f"Unexpected annotation name: {name}")



def discover_common_stems(split: str):
    """Find stems that have both image and annotation files.

    Args:
        split: Dataset split name.
    Returns:
        Sorted list of valid stems.
    Raises:
        OSError: If directory iteration fails.
    """

    img_dir = DATA_ROOT / split / "img"
    ann_dir = DATA_ROOT / split / "ann"
    img_stems = set(list_image_stems(img_dir))
    ann_stems = {annotation_path_to_stem(p) for p in ann_dir.glob("*.json")}
    return sorted(img_stems & ann_stems)


all_train_stems = discover_common_stems("train")
test_stems = discover_common_stems("test")

assert len(all_train_stems) > 0, "Không có train stems"
assert len(test_stems) > 0, "Không có test stems"

split_json_path = PROJECT_DIR / CONFIG["SPLIT_JSON_NAME"]
rng = random.Random(CONFIG["SEED"])

if CONFIG["OVERFIT_MODE"]:
    shuffled = all_train_stems.copy()
    rng.shuffle(shuffled)
    overfit_n = min(CONFIG["OVERFIT_SAMPLES"], len(shuffled))
    train_stems = sorted(shuffled[:overfit_n])
    val_stems = list(train_stems)
    print(f"OVERFIT MODE: using the same {len(train_stems)} samples for train and val")
else:
    if split_json_path.exists():
        with open(split_json_path, "r", encoding="utf-8") as f:
            split_data = json.load(f)
        train_stems = split_data["train"]
        val_stems = split_data["val"]
        print("Loaded fixed split from:", split_json_path)
    else:
        shuffled = all_train_stems.copy()
        rng.shuffle(shuffled)
        n_val = max(1, int(len(shuffled) * CONFIG["VAL_RATIO"]))
        val_stems = sorted(shuffled[:n_val])
        train_stems = sorted(shuffled[n_val:])

        split_data = {
            "seed": CONFIG["SEED"],
            "train": train_stems,
            "val": val_stems,
            "all_train_count": len(all_train_stems),
            "test_count": len(test_stems),
        }
        with open(split_json_path, "w", encoding="utf-8") as f:
            json.dump(split_data, f, ensure_ascii=False, indent=2)
        print("Saved fixed split to:", split_json_path)

print(f"train={len(train_stems)} | val={len(val_stems)} | test={len(test_stems)}")
print("train sample stems:", train_stems[:5])

Loaded fixed split from: /content/drive/MyDrive/tmp/splits_seed42.json
train=3583 | val=398 | test=2135


# Tính class weights từ mask train

In [ ]:
def load_mask_from_annotation(split: str, stem: str):
    """Build a dense mask from one JSON annotation file.

    Args:
        split: Dataset split name.
        stem: File stem without extension.
    Returns:
        Tuple of `(mask, flags)`.
    Raises:
        FileNotFoundError: If image or annotation file is missing.
    """

    img_dir = DATA_ROOT / split / "img"
    ann_dir = DATA_ROOT / split / "ann"
    img_path = find_image_path(img_dir, stem)
    ann_path = find_annotation_path(ann_dir, stem)
    if img_path is None:
        raise FileNotFoundError(f"Cannot find image for stem={stem}")

    with Image.open(img_path) as img:
        image_hw = (img.height, img.width)
    return rasterize_annotation_to_mask(ann_path, image_hw, background_id=BACKGROUND_ID, num_classes=NUM_CLASSES)



def compute_class_weights_and_presence(split: str, stems, num_classes: int, background_id: int = 0):
    """Compute class weights and per-class presence counts.

    Args:
        split: Dataset split name.
        stems: Iterable of sample stems.
        num_classes: Total number of classes including background.
        background_id: Background class id.
    Returns:
        Tuple of `(weights, pixel_counter, presence_counter, anomaly_counter)`.
    Raises:
        FileNotFoundError: If an image or annotation file is missing.
    """

    pixel_counter = np.zeros(num_classes, dtype=np.int64)
    presence_counter = np.zeros(num_classes, dtype=np.int64)
    anomaly_counter = Counter()

    for stem in tqdm(stems, desc=f"Compute weights/presence [{split}]"):
        mask, flags = load_mask_from_annotation(split, stem)
        anomaly_counter.update(flags)
        valid = mask[(mask >= 0) & (mask < num_classes)]
        counts = np.bincount(valid, minlength=num_classes)
        pixel_counter += counts
        presence_counter += (counts > 0).astype(np.int64)

    freq = pixel_counter / (pixel_counter.sum() + 1e-12)
    weights = 1.0 / np.sqrt(freq + 1e-12)
    weights[pixel_counter == 0] = 0.0
    weights[background_id] *= 0.5

    nonzero = weights[weights > 0]
    if len(nonzero) > 0:
        weights = weights / nonzero.mean()

    return torch.tensor(weights, dtype=torch.float32), pixel_counter, presence_counter, anomaly_counter


class_weights, pixel_counter, train_presence_count, train_anomalies = compute_class_weights_and_presence(
    split="train",
    stems=train_stems,
    num_classes=NUM_CLASSES,
    background_id=BACKGROUND_ID,
)

_, _, val_presence_count, val_anomalies = compute_class_weights_and_presence(
    split="train",
    stems=val_stems,
    num_classes=NUM_CLASSES,
    background_id=BACKGROUND_ID,
)

print("background weight:", float(class_weights[BACKGROUND_ID]))
print("nonzero weights:", int((class_weights > 0).sum().item()))

top_ids = np.argsort(-pixel_counter)[:10]
print("\nTop pixel classes:")
for cid in top_ids:
    cname = "background" if cid == 0 else ID_TO_CLASS.get(int(cid), f"class_{cid}")
    print(f"id={cid:>2} | {cname:<20} | pixels={int(pixel_counter[cid])}")

missing_train = [ID_TO_CLASS[i] for i in range(1, NUM_CLASSES) if train_presence_count[i] == 0]
missing_val = [ID_TO_CLASS[i] for i in range(1, NUM_CLASSES) if val_presence_count[i] == 0]
rare_train_ids = np.argsort(train_presence_count[1:])[:10] + 1

print("\nMissing classes in train:", missing_train)
print("Missing classes in val  :", missing_val)
print("\n10 lowest train presence classes:")
for cid in rare_train_ids:
    print(f"id={cid:>2} | {ID_TO_CLASS[cid]:<20} | train_presence={int(train_presence_count[cid])} | val_presence={int(val_presence_count[cid])}")

print("\nTrain anomalies:", dict(train_anomalies))
print("Val anomalies  :", dict(val_anomalies))

Compute class weights:   0%|          | 0/3583 [00:00<?, ?it/s]

background weight: 0.018276158720254898
nonzero weights: 76

Top pixel classes:
id= 0 | background           | pixels=1907418249
id=15 | chicken duck         | pixels=189535831
id= 7 | bread                | pixels=124118445
id=55 | potato               | pixels=99855039
id=54 | pork                 | pixels=98682561
id=68 | steak                | pixels=89316016
id=60 | rice                 | pixels=86573963
id= 8 | broccoli             | pixels=76503347
id=17 | corn                 | pixels=55684495
id=62 | sauce                | pixels=48077858


# Augmentation + Dataset + DataLoader

In [7]:
SIZE = CONFIG["IMAGE_SIZE"]
BG = BACKGROUND_ID

train_tf = A.Compose([
    A.LongestMaxSize(max_size=SIZE, interpolation=cv2.INTER_LINEAR),
    A.PadIfNeeded(min_height=SIZE, min_width=SIZE, border_mode=cv2.BORDER_CONSTANT, fill=0, fill_mask=BG),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(transpose_mask=False),
])

val_tf = A.Compose([
    A.LongestMaxSize(max_size=SIZE, interpolation=cv2.INTER_LINEAR),
    A.PadIfNeeded(min_height=SIZE, min_width=SIZE, border_mode=cv2.BORDER_CONSTANT, fill=0, fill_mask=BG),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(transpose_mask=False),
])


class FoodSegRebalancedDataset(Dataset):
    """Dataset wrapper for the rebalanced JSON-annotation dataset.

    Args:
        root: Dataset root path.
        split: Dataset split name.
        stems: Iterable of sample stems.
        transform: Optional Albumentations transform.
    Returns:
        Dataset instance.
    Raises:
        FileNotFoundError: If an image or annotation file is missing.
    """

    def __init__(self, root, split, stems, transform=None):
        self.root = Path(root)
        self.split = split
        self.stems = list(stems)
        self.transform = transform
        self.img_dir = self.root / split / "img"
        self.ann_dir = self.root / split / "ann"

    def __len__(self):
        """Return dataset length.

        Args:
        Returns:
            Number of samples.
        Raises:
        """

        return len(self.stems)

    def __getitem__(self, idx):
        """Load one image-mask pair.

        Args:
            idx: Sample index.
        Returns:
            Dictionary containing image tensor, mask tensor, stem, and flags.
        Raises:
            FileNotFoundError: If an image or annotation file is missing.
        """

        stem = self.stems[idx]
        img_path = find_image_path(self.img_dir, stem)
        ann_path = find_annotation_path(self.ann_dir, stem)

        if img_path is None:
            raise FileNotFoundError(f"Không thấy ảnh cho stem={stem}")

        image = np.array(Image.open(img_path).convert("RGB"))
        mask, ann_flags = rasterize_annotation_to_mask(
            ann_path=ann_path,
            image_hw=(image.shape[0], image.shape[1]),
            background_id=BACKGROUND_ID,
            num_classes=NUM_CLASSES,
        )

        if self.transform is not None:
            aug = self.transform(image=image, mask=mask)
            image = aug["image"]
            mask = aug["mask"]

        if not torch.is_tensor(mask):
            mask = torch.tensor(mask, dtype=torch.long)
        else:
            mask = mask.long()

        return {
            "image": image,
            "mask": mask,
            "stem": stem,
            "ann_flags": ann_flags,
        }


train_ds = FoodSegRebalancedDataset(DATA_ROOT, "train", train_stems, transform=train_tf)
val_ds   = FoodSegRebalancedDataset(DATA_ROOT, "train", val_stems, transform=val_tf)
test_ds  = FoodSegRebalancedDataset(DATA_ROOT, "test", test_stems, transform=val_tf)

train_loader = DataLoader(
    train_ds,
    batch_size=CONFIG["BATCH_SIZE"],
    shuffle=True,
    num_workers=CONFIG["NUM_WORKERS"],
    pin_memory=CONFIG["PIN_MEMORY"],
    drop_last=False,
    persistent_workers=(CONFIG["NUM_WORKERS"] > 0),
)

val_loader = DataLoader(
    val_ds,
    batch_size=CONFIG["BATCH_SIZE"],
    shuffle=False,
    num_workers=CONFIG["NUM_WORKERS"],
    pin_memory=CONFIG["PIN_MEMORY"],
    drop_last=False,
    persistent_workers=(CONFIG["NUM_WORKERS"] > 0),
)

print("train_ds:", len(train_ds))
print("val_ds:", len(val_ds))
print("test_ds:", len(test_ds))

sample = train_ds[0]
print("image shape:", tuple(sample["image"].shape))
print("mask shape :", tuple(sample["mask"].shape))
print("mask min/max:", int(sample["mask"].min()), int(sample["mask"].max()))
print("ann flags:", sample["ann_flags"][:5])

train_ds: 3583
val_ds: 398
test_ds: 2135
image shape: (3, 384, 384)
mask shape : (384, 384)
mask min/max: 0 62


# MODEL
BiSeNet + ResNet18 pretrained

In [8]:
# =========================
# MODEL - CHANGED
# BiSeNetV1 + ResNet18 pretrained backbone
# =========================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18, ResNet18_Weights

class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class SpatialPath(nn.Module):
    # output stride 8
    def __init__(self, out_ch=128):
        super().__init__()
        self.conv1 = ConvBNReLU(3, 64, k=7, s=2, p=3)   # /2
        self.conv2 = ConvBNReLU(64, 64, k=3, s=2, p=1)  # /4
        self.conv3 = ConvBNReLU(64, 64, k=3, s=2, p=1)  # /8
        self.conv_out = ConvBNReLU(64, out_ch, k=1, s=1, p=0)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv_out(x)
        return x

class AttentionRefinementModule(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = ConvBNReLU(in_ch, out_ch, k=3, s=1, p=1)
        self.attn = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(out_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.Sigmoid(),
        )

    def forward(self, x):
        feat = self.conv(x)
        attn = self.attn(feat)
        return feat * attn

class FeatureFusionModule(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.convblk = ConvBNReLU(in_ch, out_ch, k=1, s=1, p=0)
        self.attn = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(out_ch, out_ch // 4, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch // 4, out_ch, kernel_size=1, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, fsp, fcp):
        feat = torch.cat([fsp, fcp], dim=1)
        feat = self.convblk(feat)
        attn = self.attn(feat)
        feat_attn = feat * attn
        feat_out = feat + feat_attn
        return feat_out

class SegHead(nn.Module):
    def __init__(self, in_ch, mid_ch, n_classes):
        super().__init__()
        self.conv = ConvBNReLU(in_ch, mid_ch, k=3, s=1, p=1)
        self.dropout = nn.Dropout2d(0.1)
        self.cls = nn.Conv2d(mid_ch, n_classes, kernel_size=1)

    def forward(self, x, out_size):
        x = self.conv(x)
        x = self.dropout(x)
        x = self.cls(x)
        x = F.interpolate(x, size=out_size, mode="bilinear", align_corners=False)
        return x

class ContextPathResNet18(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = resnet18(weights=weights)

        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool

        self.layer1 = backbone.layer1   # /4
        self.layer2 = backbone.layer2   # /8
        self.layer3 = backbone.layer3   # /16
        self.layer4 = backbone.layer4   # /32

        self.arm16 = AttentionRefinementModule(256, 128)
        self.arm32 = AttentionRefinementModule(512, 128)

        self.global_context_conv = ConvBNReLU(512, 128, k=1, s=1, p=0)
        self.conv_head16 = ConvBNReLU(128, 128, k=3, s=1, p=1)
        self.conv_head32 = ConvBNReLU(128, 128, k=3, s=1, p=1)

    def forward(self, x):
        x = self.conv1(x)     # /2
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)   # /4

        x = self.layer1(x)    # /4
        feat8 = self.layer2(x)   # /8
        feat16 = self.layer3(feat8)  # /16
        feat32 = self.layer4(feat16) # /32

        gc = F.adaptive_avg_pool2d(feat32, 1)
        gc = self.global_context_conv(gc)
        gc = F.interpolate(gc, size=feat32.shape[-2:], mode="bilinear", align_corners=False)

        feat32_arm = self.arm32(feat32)
        feat32_sum = feat32_arm + gc
        feat32_up = F.interpolate(feat32_sum, size=feat16.shape[-2:], mode="bilinear", align_corners=False)
        feat32_up = self.conv_head32(feat32_up)

        feat16_arm = self.arm16(feat16)
        feat16_sum = feat16_arm + feat32_up
        feat16_up = F.interpolate(feat16_sum, size=feat8.shape[-2:], mode="bilinear", align_corners=False)
        feat16_up = self.conv_head16(feat16_up)

        # return context at /8 and aux source /16
        return feat8, feat16_up, feat32_up

class BiSeNetResNet18(nn.Module):
    def __init__(self, num_classes=76, pretrained_backbone=True, use_aux=True):
        super().__init__()
        self.use_aux = use_aux

        self.spatial_path = SpatialPath(out_ch=128)
        self.context_path = ContextPathResNet18(pretrained=pretrained_backbone)
        self.ffm = FeatureFusionModule(in_ch=128 + 128, out_ch=256)

        self.head = SegHead(256, 256, num_classes)

        if self.use_aux:
            self.aux_head16 = SegHead(128, 128, num_classes)
            self.aux_head32 = SegHead(128, 128, num_classes)

    def forward(self, x):
        out_size = x.shape[-2:]

        feat_sp = self.spatial_path(x)
        feat8, feat_cp8, feat_cp16 = self.context_path(x)

        feat_fuse = self.ffm(feat_sp, feat_cp8)
        logits = self.head(feat_fuse, out_size)

        if self.training and self.use_aux:
            aux16 = self.aux_head16(feat_cp8, out_size)
            aux32 = self.aux_head32(feat_cp16, out_size)
            return logits, aux16, aux32

        return logits

checkpoint helpers

In [9]:
# =========================
# CHECKPOINT HELPERS - CHANGED
# Giữ tối thiểu: latest + best
# Không lưu periodic để đỡ tốn disk/RAM serialize
# =========================
from pathlib import Path
import torch
import pandas as pd

PROJECT_DIR = Path(CONFIG["PROJECT_DIR"])
CKPT_DIR = PROJECT_DIR / CONFIG["CKPT_DIRNAME"]
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

def get_latest_checkpoint(ckpt_dir: Path):
    latest = ckpt_dir / "latest.pth"
    return latest if latest.exists() else None

def save_checkpoint(state: dict, ckpt_dir: Path, is_best: bool = False):
    torch.save(state, ckpt_dir / "latest.pth")
    if is_best and CONFIG["SAVE_BEST"]:
        torch.save(state, ckpt_dir / "best.pth")

def append_log_row(log_path: Path, row: dict):
    df = pd.DataFrame([row])
    if log_path.exists():
        df.to_csv(log_path, mode="a", header=False, index=False)
    else:
        df.to_csv(log_path, index=False)

# train/val functions

In [10]:
# =========================
# TRAIN / VAL FUNCTIONS - CHANGED
# phù hợp output của BiSeNet ResNet18 pretrained
# =========================
from tqdm.auto import tqdm
import torch
import torch.nn.functional as F

@torch.no_grad()
def update_hist(hist, pred, target, num_classes):
    mask = (target >= 0) & (target < num_classes)
    x = num_classes * target[mask] + pred[mask]
    binc = torch.bincount(x.reshape(-1), minlength=num_classes ** 2)
    hist += binc.reshape(num_classes, num_classes)
    return hist

@torch.no_grad()
def compute_metrics_from_hist(hist):
    hist = hist.float()
    eps = 1e-6
    tp = torch.diag(hist)
    gt = hist.sum(dim=1)
    pred = hist.sum(dim=0)

    iou = tp / (gt + pred - tp + eps)
    acc_cls = tp / (gt + eps)
    aacc = tp.sum() / (hist.sum() + eps)

    return {
        "mIoU": torch.nanmean(iou).item(),
        "mAcc": torch.nanmean(acc_cls).item(),
        "aAcc": aacc.item(),
    }

def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, device, epoch):
    model.train()
    total_loss = 0.0

    pbar = tqdm(loader, desc=f"Epoch {epoch:03d} [Train]", leave=False)
    for batch in pbar:
        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True).long()

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(device.type == "cuda" and CONFIG["USE_AMP"])):
            outputs = model(images)

            if CONFIG["USE_AUX_HEAD"]:
                main_logits, aux16, aux32 = outputs
                loss_main = criterion(main_logits, masks)
                loss_aux16 = criterion(aux16, masks)
                loss_aux32 = criterion(aux32, masks)
                loss = loss_main + CONFIG["AUX_WEIGHT"] * (loss_aux16 + loss_aux32) * 0.5
            else:
                main_logits = outputs
                loss = criterion(main_logits, masks)

        scaler.scale(loss).backward()

        if CONFIG["GRAD_CLIP_NORM"] is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["GRAD_CLIP_NORM"])

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += float(loss.item())
        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{optimizer.param_groups[0]['lr']:.6f}")

        del images, masks, outputs, loss
        if device.type == "cuda":
            torch.cuda.empty_cache()

    return total_loss / max(1, len(loader))

@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device, num_classes, epoch):
    model.eval()
    total_loss = 0.0
    hist = torch.zeros((num_classes, num_classes), device=device, dtype=torch.int64)

    pbar = tqdm(loader, desc=f"Epoch {epoch:03d} [Val]", leave=False)
    for batch in pbar:
        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True).long()

        with torch.amp.autocast("cuda", enabled=(device.type == "cuda" and CONFIG["USE_AMP"])):
            logits = model(images)
            loss = criterion(logits, masks)

        total_loss += float(loss.item())
        preds = torch.argmax(logits, dim=1)
        hist = update_hist(hist, preds, masks, num_classes)

        del images, masks, logits, preds
        if device.type == "cuda":
            torch.cuda.empty_cache()

    metrics = compute_metrics_from_hist(hist)
    return total_loss / max(1, len(loader)), metrics

init model / optimizer / resume

In [11]:
# =========================
# INIT MODEL / OPT / RESUME - CHANGED
# =========================
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("OVERFIT_MODE:", CONFIG["OVERFIT_MODE"])
print("NUM_CLASSES:", NUM_CLASSES)

model = BiSeNetResNet18(
    num_classes=NUM_CLASSES,
    pretrained_backbone=CONFIG["PRETRAINED_BACKBONE"],
    use_aux=CONFIG["USE_AUX_HEAD"],
).to(device)

# Overfit debug should stay simple: plain CE is easier to diagnose.
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=CONFIG["LR"],
    momentum=CONFIG["MOMENTUM"],
    weight_decay=CONFIG["WEIGHT_DECAY"],
)

total_iters = CONFIG["EPOCHS"] * max(1, len(train_loader))
poly_lambda = lambda it: (1 - min(it, total_iters) / max(1, total_iters)) ** CONFIG["POLY_POWER"]
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=poly_lambda)

scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda" and CONFIG["USE_AMP"]))

start_epoch = 1
best_miou = -1.0
global_iter = 0

if CONFIG["RESUME"]:
    latest_ckpt = get_latest_checkpoint(CKPT_DIR)
else:
    latest_ckpt = None

if CONFIG["RESUME"] and latest_ckpt is not None:
    print("Resuming from:", latest_ckpt)
    ckpt = torch.load(latest_ckpt, map_location=device)

    model.load_state_dict(ckpt["model_state"], strict=True)
    optimizer.load_state_dict(ckpt["optimizer_state"])

    if ckpt.get("scheduler_state") is not None:
        scheduler.load_state_dict(ckpt["scheduler_state"])
    if ckpt.get("scaler_state") is not None and device.type == "cuda":
        scaler.load_state_dict(ckpt["scaler_state"])

    start_epoch = int(ckpt["epoch"]) + 1
    best_miou = float(ckpt.get("best_miou", -1.0))
    global_iter = int(ckpt.get("global_iter", 0))

    print(f"Resume OK -> start_epoch={start_epoch}, best_miou={best_miou:.4f}")
else:
    print("No checkpoint found. Train from scratch.")

batch = next(iter(train_loader))
assert int(batch["mask"].min()) >= 0, "Mask contains negative labels"
assert int(batch["mask"].max()) < NUM_CLASSES, f"Mask max must be < {NUM_CLASSES}"

print("sanity image:", tuple(batch["image"].shape))
print("sanity mask :", tuple(batch["mask"].shape))
print("mask min/max:", int(batch["mask"].min()), int(batch["mask"].max()))
print("unique labels in first batch:", sorted(torch.unique(batch["mask"]).cpu().tolist())[:20])
print("stems in first batch:", batch["stem"][:4])

device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 193MB/s]


Resuming from: /content/drive/MyDrive/tmp/checkpoints/latest.pth
Resume OK -> start_epoch=15, best_miou=0.1620
sanity image: (4, 3, 384, 384)
sanity mask : (4, 384, 384)
mask min/max: 0 73
